# UGUIDU Museum Eval — Word Document Ground Truth

Evaluates the UGUIDU voice assistant pipeline using `.docx` test reports as ground truth.

**Pipeline:**
1. Read Word docs from `C:\Users\jimal\Music\` — one per artifact
2. Claude extracts structured JSON from each doc
3. Live pipeline test: send each artifact label to `/api/chat`
4. Claude judge scores each RAG response against doc ground truth
5. Direct DB verification via `knowledge_base`
6. Statistical summary + export to CSV/Excel
7. Langfuse dataset creation

## Cell 1 — Install Dependencies

In [1]:
import subprocess
subprocess.run([
    'pip', 'install', '--quiet',
    'python-docx', 'langfuse', 'anthropic', 'openai',
    'psycopg2-binary', 'pandas', 'openpyxl',
    'python-dotenv', 'requests'
], check=True)
print('All dependencies installed')

All dependencies installed


## Cell 2 — Config & Connections

In [9]:
import os, re, json, time, uuid, logging
from pathlib import Path
from datetime import datetime

import requests
import pandas as pd
from dotenv import load_dotenv

# ── Load .env.local ───────────────────────────────────────────
ENV_PATH = Path('museum-deployment/.env.local')
if not ENV_PATH.exists():
    ENV_PATH = Path('.env.local')
load_dotenv(dotenv_path=ENV_PATH, override=True)

ANTHROPIC_API_KEY   = os.environ['ANTHROPIC_API_KEY']
LANGFUSE_PUBLIC_KEY = os.environ['LANGFUSE_PUBLIC_KEY']
LANGFUSE_SECRET_KEY = os.environ['LANGFUSE_SECRET_KEY']
LANGFUSE_HOST       = os.environ.get('LANGFUSE_BASE_URL', 'https://cloud.langfuse.com')
OPENAI_API_KEY      = os.environ.get('OPENAI_API_KEY', '')

# ── Set Langfuse env vars BEFORE importing get_client ────────
os.environ['LANGFUSE_PUBLIC_KEY'] = LANGFUSE_PUBLIC_KEY
os.environ['LANGFUSE_SECRET_KEY'] = LANGFUSE_SECRET_KEY
os.environ['LANGFUSE_BASE_URL']   = LANGFUSE_HOST

# ── Backend endpoints ─────────────────────────────────────────
CHAT_URL     = 'http://194.171.191.226:3414/api/chat'
CLASSIFY_URL = 'http://194.171.191.226:3414/api/classify'

# ── DB ────────────────────────────────────────────────────────
DB_HOST     = '194.171.191.227'
DB_PORT     = 5432
DB_NAME     = 'o2'
DB_USER     = 'postgres'
DB_PASSWORD = 'postgres'

# ── Word docs dir ─────────────────────────────────────────────
DOCS_DIR = Path(r'C:\Users\jimal\Music')

# ── Output dir ───────────────────────────────────────────────
OUTPUT_DIR = Path('eval_output')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Claude model ─────────────────────────────────────────────
CLAUDE_MODEL = 'claude-sonnet-4-20250514'

# ── LABEL_TO_DB ───────────────────────────────────────────────
LABEL_TO_DB = {
    'Justinus of Nassau': 'Justinus van Nassau',
    'Frederick Henry of Orange-Nassau': 'Frederik Hendrik van Nassau',
    'Maurice of Oranje-Nassau': 'Maurits van Nassau',
    'Equestrian statue of William of Orange': 'ruiterstandbeeld Willem',
    'Money of necessity minted during the siege of breda': 'Noodmunt Breda',
    'Noodmunten': 'Noodmunt Breda',
    'City map of Breda in relief': 'Stadsplan reliëf',
    'Model of a 24-pound gun': 'kanon Breda',
    'Constantijn Huygens, curator van de Illustere School in Breda': 'Constantijn Huygens',
    'Beeld vna de turfschipper': 'turfschipper',
    'Departure of the Spanish occupation from Breda on 10 October 1637': 'Uittocht Spaanse bezetting',
    'Retreat of the Spanish Garrison from Breda': 'Uittocht Spaanse garnizoen',
    'Military Operations in flanders with siege and capture of breda': 'beleg van Breda Spinola',
    'Herdenkingsbeker van het beleg van Breda in 1637': 'Herdenkingsbeker beleg Breda',
    'Obsidio Bredana': 'Obsidio Bredana',
    'het beleg van Breda': 'beleg van Breda',
    'Model van het Turfschip': 'Turfschip van Breda',
    'Turfschip van Breda in 1590': 'Turfschip van Breda',
    'Radslotpistool van Charles de Heraughières, ca. 1590-1600': 'Radslotpistool',
    'Schelpdieren en vis': 'Schelpdieren',
    'partizaan, 17de eeuw, metaal': 'Partizaan',
    'de overgave van breda': 'overgave van Breda',
    'collectie 17de en 18de eeuws metaal militair tuig': 'militair tuig',
    'Sporen van de Nassaus': 'Sporen Nassaus',
    'Portret van Jan de Jongere, graaf van nassau Siegen': 'Jan van Nassau',
    'portret van lodewijk van Nassau': 'Lodewijk van Nassau',
    'Filips Willem, prins van Oranje': 'Filips Willem',
    'Isabella Clara Eugenia van Spanja': 'Isabella Clara Eugenia',
    'Het overlijden van Prins Maurits': 'overlijden Prins Maurits',
    'Albrecht van Oostenrijk': 'Albrecht van Oostenrijk',
    'Ambrogio Spinola': 'Ambrogio Spinola',
    'Frederik Hendrik, Prins van Oranje': 'Frederik Hendrik van Nassau',
}

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[logging.StreamHandler()]
)
log = logging.getLogger('eval-docs')

# ── Initialize Langfuse v4 ────────────────────────────────────
from langfuse import get_client
lf = get_client()
try:
    lf.auth_check()
    print('Langfuse: OK')
except Exception as e:
    print(f'Langfuse: WARN — {e}')

# ── Initialize Anthropic ──────────────────────────────────────
import anthropic
claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
try:
    _ping = claude.messages.create(
        model=CLAUDE_MODEL, max_tokens=5,
        messages=[{'role': 'user', 'content': 'ping'}]
    )
    print(f'Anthropic ({CLAUDE_MODEL}): OK')
except Exception as e:
    print(f'Anthropic: ERROR — {e}')

# ── Initialize DB ─────────────────────────────────────────────
import psycopg2

def get_db():
    return psycopg2.connect(
        host=DB_HOST, port=DB_PORT, dbname=DB_NAME,
        user=DB_USER, password=DB_PASSWORD, connect_timeout=10
    )

try:
    _conn = get_db()
    with _conn.cursor() as _cur:
        _cur.execute("SELECT COUNT(*) FROM knowledge_base")
        _total = _cur.fetchone()[0]
    _conn.close()
    print(f'PostgreSQL: OK — {_total} total records in knowledge_base')
except Exception as e:
    print(f'PostgreSQL: ERROR — {e}')

print(f'\nConfig ready | {len(LABEL_TO_DB)} artifact labels | Output: {OUTPUT_DIR}')

Langfuse: OK


C:\Users\jimal\AppData\Local\Temp\ipykernel_47648\1897754722.py:103: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  _ping = claude.messages.create(


Anthropic (claude-sonnet-4-20250514): OK
PostgreSQL: OK — 32265 total records in knowledge_base

Config ready | 32 artifact labels | Output: eval_output


## Cell 3 — Read All Word Documents

In [3]:
from docx import Document

# Filename stem → best-match LABEL_TO_DB key (manual mapping for edge cases)
FILENAME_TO_LABEL = {
    'Justinus of Nassau': 'Justinus of Nassau',
    'Frederick Henry of Orange': 'Frederick Henry of Orange-Nassau',
    'Maurice of Oranje': 'Maurice of Oranje-Nassau',
    'City map of Breda in relief': 'City map of Breda in relief',
    'collectie 17de en 18de eeuws metaal militair tuig': 'collectie 17de en 18de eeuws metaal militair tuig',
    'Constantijn Huygens': 'Constantijn Huygens, curator van de Illustere School in Breda',
    'De overgave van breda': 'de overgave van breda',
    'Departure of the Spanish occupation from Breda on 10 October 1637': 'Departure of the Spanish occupation from Breda on 10 October 1637',
    'Filips Willem, prins van Oranje': 'Filips Willem, prins van Oranje',
    'Frederik Hendrik, Prins van Oranje': 'Frederik Hendrik, Prins van Oranje',
    'Herdenkingsbeker van het beleg van Breda in 1637': 'Herdenkingsbeker van het beleg van Breda in 1637',
    'het beleg van Breda': 'het beleg van Breda',
    'Het overlijden van Prins Maurits': 'Het overlijden van Prins Maurits',
    'Isabella Clara Eugenia van Spanja': 'Isabella Clara Eugenia van Spanja',
    'Military Operations in flanders with siege and capture of breda': 'Military Operations in flanders with siege and capture of breda',
    'Model of a 24': 'Model of a 24-pound gun',
    'Model van het Turfschip': 'Model van het Turfschip',
    'Money of necessity minted during the siege of breda': 'Money of necessity minted during the siege of breda',
    'Noodmunten': 'Noodmunten',
    'Obsidio Bredana': 'Obsidio Bredana',
    'ONZEKER': 'ONZEKER',
    'partizaan': 'partizaan, 17de eeuw, metaal',
    'Portret van Jan de Jongere': 'Portret van Jan de Jongere, graaf van nassau Siegen',
    'portret van lodewijk van Nassau': 'portret van lodewijk van Nassau',
    'Radslotpistool van Charles de Heraughières': 'Radslotpistool van Charles de Heraughières, ca. 1590-1600',
    'Register van het meselaarsgilde met omgekomen burgers bij de Furie van Houtepen': 'Register van het meselaarsgilde met omgekomen burgers bij de Furie van Houtepen',
    'Retreat of the Spanish Garrison from Breda': 'Retreat of the Spanish Garrison from Breda',
    'Schelpdieren en vis': 'Schelpdieren en vis',
    'Beeld van de turfschipper': 'Beeld vna de turfschipper',
    'Ambrogio Spinola': 'Ambrogio Spinola',
}

def read_docx(path: Path) -> str:
    doc = Document(str(path))
    paras = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    # Also extract tables
    for table in doc.tables:
        for row in table.rows:
            cells = [c.text.strip() for c in row.cells if c.text.strip()]
            if cells:
                paras.append(' | '.join(cells))
    return '\n'.join(paras)

docx_files = sorted(DOCS_DIR.glob('*.docx'))
doc_store = {}  # artifact_label → full_text
skipped = []

for f in docx_files:
    stem = f.stem
    label = FILENAME_TO_LABEL.get(stem)
    if label is None:
        # Fallback: try case-insensitive key match
        for k in LABEL_TO_DB:
            if k.lower() == stem.lower():
                label = k
                break
    if label is None:
        skipped.append(stem)
        label = stem  # keep with original name
    try:
        text = read_docx(f)
        doc_store[label] = text
    except Exception as e:
        log.warning(f'Failed to read {f.name}: {e}')

print(f'Read {len(doc_store)} Word documents from {DOCS_DIR}\n')
print(f'{"Label":<60} {"Chars":>6}')
print('-' * 70)
for lbl, txt in sorted(doc_store.items()):
    print(f'{lbl[:58]:<60} {len(txt):>6}')
if skipped:
    print(f'\nUnmapped filenames (kept as-is): {skipped}')

Read 30 Word documents from C:\Users\jimal\Music

Label                                                         Chars
----------------------------------------------------------------------
Ambrogio Spinola                                               1576
Beeld vna de turfschipper                                      1003
City map of Breda in relief                                    2483
Constantijn Huygens, curator van de Illustere School in Br     5594
Departure of the Spanish occupation from Breda on 10 Octob     4775
Filips Willem, prins van Oranje                                7170
Frederick Henry of Orange-Nassau                               5494
Frederik Hendrik, Prins van Oranje                             6556
Herdenkingsbeker van het beleg van Breda in 1637               8054
Het overlijden van Prins Maurits                               7117
Isabella Clara Eugenia van Spanja                              5834
Justinus of Nassau                                             

## Cell 4 — Extract Structured Data with Claude

In [14]:
def extract_structured(label: str, doc_text: str) -> dict:
    try:
        resp = claude.messages.create(
            model='claude-sonnet-4-5',  # updated model
            max_tokens=1024,
            messages=[{'role': 'user', 'content': f'Artifact label: {label}\n\nDocument text:\n{doc_text[:6000]}\n\n{EXTRACT_PROMPT}'}]
        )
        raw = resp.content[0].text.strip()
        raw = re.sub(r'^```(?:json)?|```$', '', raw, flags=re.MULTILINE).strip()
        result = json.loads(raw)
        log.info(f'Extracted {label[:40]}: rag_quality={result.get("rag_quality")} facts={len(result.get("key_facts",[]))}')
        return result
    except Exception as e:
        log.warning(f'Extract failed for {label}: {e}')
        return {
            'artifact_name': label,
            'detection_rate': 'unknown',
            'rag_quality': 'missing',
            'db_records_found': False,
            'key_facts': [],
            'main_issues': [f'Extraction error: {e}'],
            'recommendations': [],
            'llm_response_quality': 'wrong',
            'ragtruth_label': 'evident_baseless'
        }

In [12]:
import json
from pathlib import Path

# Delete corrupted cache
cache_path = Path('eval_output/extract_cache.json')
cache_path.unlink()
print('Cache deleted — rerun Cell 4')

Cache deleted — rerun Cell 4


## Cell 5 — Live Pipeline Test

In [6]:
import langfuse
print(langfuse.__version__)

4.6.1


In [8]:
from langfuse import get_client

# Replace lf = Langfuse(...) with:
lf = get_client()

# Replace lf.trace(...) with:
def query_pipeline(cv_label: str, audience: str = 'tourist') -> str:
    payload = {
        'message': cv_label,
        'landmark': cv_label,
        'audience': audience,
        'conversation_id': str(uuid.uuid4()),
    }
    db_term = LABEL_TO_DB.get(cv_label, cv_label)

    with lf.start_as_current_observation(
        name='eval-live-rag-test',
        as_type='span',
        input={'artifact_name': cv_label, 'db_search_term': db_term}
    ) as trace:
        for attempt in range(3):
            parts = []
            try:
                with requests.post(CHAT_URL, json=payload, stream=True, timeout=90) as resp:
                    resp.raise_for_status()
                    for raw in resp.iter_lines():
                        if not raw:
                            continue
                        line = raw.decode('utf-8') if isinstance(raw, bytes) else raw
                        try:
                            obj = json.loads(line)
                            if obj.get('type') == 'answer':
                                parts.append(obj.get('content', ''))
                        except json.JSONDecodeError:
                            pass
                response_text = ''.join(parts).strip()
                trace.update(output={'rag_response': response_text})
                return response_text
            except Exception as e:
                wait = 2 ** attempt
                log.warning(f'API attempt {attempt+1} failed: {e}')
                time.sleep(wait)

        trace.update(output={'error': 'API failed after 3 attempts'})
        return 'ERROR: API failed after 3 attempts'

## Cell 6 — Claude Judge Evaluation

In [ ]:
JUDGE_SYSTEM = """You are a strict evaluator for a museum voice assistant RAG pipeline.
GROUND TRUTH comes from expert Word documents — it is the sole reference for facts.
Score the RAG_RESPONSE against ground truth. Output JSON only."""

JUDGE_PROMPT = """ARTIFACT: {artifact_name}

GROUND_TRUTH_KEY_FACTS (from Word doc):
{key_facts}

RAG_RESPONSE (from live pipeline):
{rag_response}

Score the RAG_RESPONSE:
- factual_accuracy: 0-3 (3=all facts correct, 2=mostly correct, 1=partial, 0=wrong/irrelevant)
- hallucination: true if response states facts NOT in ground truth, else false
- rag_retrieved_correct: true if response is about the right artifact, else false
- main_issue: one sentence describing the biggest problem (or "none" if correct)
- recommendation: one specific fix (or "none" if correct)

Return EXACTLY this JSON (no markdown):
{{"factual_accuracy":0,"hallucination":false,"rag_retrieved_correct":false,"main_issue":"string","recommendation":"string"}}"""

def judge_response(artifact_name: str, key_facts: list, rag_response: str) -> dict:
    if rag_response.startswith('ERROR'):
        return {
            'factual_accuracy': 0, 'hallucination': False,
            'rag_retrieved_correct': False,
            'main_issue': 'API error — no response to judge',
            'recommendation': 'Fix API connectivity'
        }

    facts_str = '\n'.join(f'- {f}' for f in key_facts) if key_facts else '(no key facts extracted from doc)'
    prompt = JUDGE_PROMPT.format(
        artifact_name=artifact_name,
        key_facts=facts_str,
        rag_response=rag_response[:2000]
    )

    trace = lf.trace(
        name='eval-judge',
        input={'ground_truth_facts': key_facts, 'rag_response': rag_response[:500]},
        metadata={'artifact_name': artifact_name}
    )
    generation = trace.generation(
        name='claude-judge',
        model=CLAUDE_MODEL,
        input=[{'role': 'user', 'content': prompt}]
    )

    try:
        resp = claude.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=512,
            system=JUDGE_SYSTEM,
            messages=[{'role': 'user', 'content': prompt}]
        )
        raw = resp.content[0].text.strip()
        raw = re.sub(r'^```(?:json)?|```$', '', raw, flags=re.MULTILINE).strip()
        scores = json.loads(raw)
        generation.end(output=scores)
        trace.update(output=scores)

        # Log scores to Langfuse
        lf.score(
            trace_id=trace.id,
            name='factual_accuracy',
            value=scores.get('factual_accuracy', 0),
            data_type='NUMERIC'
        )
        lf.score(
            trace_id=trace.id,
            name='hallucination',
            value=1 if scores.get('hallucination') else 0,
            data_type='NUMERIC'
        )
        return scores
    except Exception as e:
        generation.end(output={'error': str(e)}, level='ERROR')
        log.warning(f'Judge failed for {artifact_name}: {e}')
        return {
            'factual_accuracy': 0, 'hallucination': False,
            'rag_retrieved_correct': False,
            'main_issue': f'Judge error: {e}',
            'recommendation': 'Retry'
        }

# Run judge for all labels
JUDGE_CACHE_PATH = OUTPUT_DIR / 'judge_scores.json'
if JUDGE_CACHE_PATH.exists():
    with open(JUDGE_CACHE_PATH, encoding='utf-8') as f:
        judge_scores = json.load(f)
    print(f'Loaded {len(judge_scores)} cached judge scores')
else:
    judge_scores = {}

for lbl in test_labels:
    if lbl in judge_scores:
        continue
    ext = extracted.get(lbl, {})
    key_facts = ext.get('key_facts', [])
    rag_resp = live_responses.get(lbl, 'ERROR: no response')
    log.info(f'Judging: {lbl[:50]}')
    judge_scores[lbl] = judge_response(lbl, key_facts, rag_resp)
    time.sleep(0.3)

with open(JUDGE_CACHE_PATH, 'w', encoding='utf-8') as f:
    json.dump(judge_scores, f, ensure_ascii=False, indent=2)

print(f'\nJudge scores for {len(judge_scores)} artifacts:\n')
print(f'{"Label":<55} {"FA":>3} {"Halluc":>6} {"RAG OK":>7}')
print('-' * 75)
for lbl, s in sorted(judge_scores.items()):
    fa  = s.get('factual_accuracy', '?')
    hal = 'YES' if s.get('hallucination') else 'no'
    rag = 'YES' if s.get('rag_retrieved_correct') else 'no'
    print(f'{lbl[:53]:<55} {str(fa):>3} {hal:>6} {rag:>7}')

## Cell 7 — DB Verification

In [ ]:
def db_search(search_term: str, limit: int = 5) -> list:
    conn = get_db()
    try:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT identifier, title, description, content
                FROM knowledge_base
                WHERE title ILIKE %s
                LIMIT %s
            """, (f'%{search_term}%', limit))
            rows = cur.fetchall()
            return [{'identifier': r[0], 'title': r[1],
                     'description': (r[2] or '')[:200],
                     'content_len': len(r[3] or '')} for r in rows]
    finally:
        conn.close()

def classify_coverage(rows: list) -> str:
    if not rows:
        return 'missing'
    avg_content = sum(r['content_len'] for r in rows) / len(rows)
    if avg_content > 300:
        return 'rich'
    if avg_content > 50:
        return 'sparse'
    return 'sparse'

db_coverage = {}  # label → {coverage, rows, search_term}

for lbl, db_term in LABEL_TO_DB.items():
    rows = db_search(db_term)
    coverage = classify_coverage(rows)
    db_coverage[lbl] = {'search_term': db_term, 'coverage': coverage, 'rows': rows}

print(f'DB Coverage for {len(db_coverage)} artifacts:\n')
print(f'{"Label":<55} {"DB Term":<30} {"Coverage":<10} {"Hits":>5}')
print('-' * 105)
for lbl, info in sorted(db_coverage.items(), key=lambda x: x[1]['coverage']):
    hits = len(info['rows'])
    cov  = info['coverage'].upper()
    term = info['search_term'][:28]
    print(f'{lbl[:53]:<55} {term:<30} {cov:<10} {hits:>5}')

# Summary counts
rich    = sum(1 for v in db_coverage.values() if v['coverage'] == 'rich')
sparse  = sum(1 for v in db_coverage.values() if v['coverage'] == 'sparse')
missing = sum(1 for v in db_coverage.values() if v['coverage'] == 'missing')
print(f'\nRich: {rich}  |  Sparse: {sparse}  |  Missing: {missing}')

## Cell 8 — Statistical Summary

In [ ]:
# Build combined DataFrame
rows = []
for lbl in test_labels:
    ext  = extracted.get(lbl, {})
    sc   = judge_scores.get(lbl, {})
    cov  = db_coverage.get(lbl, {})
    resp = live_responses.get(lbl, '')
    rows.append({
        'artifact_label':       lbl,
        'db_search_term':       LABEL_TO_DB.get(lbl, ''),
        'doc_artifact_name':    ext.get('artifact_name', lbl),
        'detection_rate':       ext.get('detection_rate', 'unknown'),
        'doc_rag_quality':      ext.get('rag_quality', 'unknown'),
        'doc_ragtruth_label':   ext.get('ragtruth_label', 'unknown'),
        'llm_response_quality': ext.get('llm_response_quality', 'unknown'),
        'key_facts_count':      len(ext.get('key_facts', [])),
        'issues_count':         len(ext.get('main_issues', [])),
        'recommendations_count':len(ext.get('recommendations', [])),
        'factual_accuracy':     sc.get('factual_accuracy', 0),
        'hallucination':        1 if sc.get('hallucination') else 0,
        'rag_retrieved_correct':1 if sc.get('rag_retrieved_correct') else 0,
        'main_issue':           sc.get('main_issue', ''),
        'recommendation':       sc.get('recommendation', ''),
        'db_coverage':          cov.get('coverage', 'missing'),
        'db_hits':              len(cov.get('rows', [])),
        'rag_response_preview': (resp or '')[:300],
        'has_doc':              lbl in doc_store,
    })

df = pd.DataFrame(rows)

total = len(df)
SEP = '=' * 65
print(SEP)
print('UGUIDU EVALUATION SUMMARY — WORD DOC GROUND TRUTH')
print(f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M")} | Artifacts: {total}')
print(SEP)

# 1. Overall accuracy
avg_fa   = df['factual_accuracy'].mean()
hal_rate = df['hallucination'].mean() * 100
rag_rate = df['rag_retrieved_correct'].mean() * 100
pass_n   = (df['factual_accuracy'] >= 2).sum()
print(f'\n--- 1. OVERALL ACCURACY ---')
print(f'  Mean factual accuracy:       {avg_fa:.2f} / 3.0')
print(f'  Pass rate (FA >= 2):         {pass_n}/{total} ({pass_n/total*100:.1f}%)')
print(f'  Hallucination rate:          {hal_rate:.1f}%  ({int(df["hallucination"].sum())}/{total})')
print(f'  RAG retrieval correct rate:  {rag_rate:.1f}%  ({int(df["rag_retrieved_correct"].sum())}/{total})')

# 2. FA distribution
print(f'\n--- 2. FACTUAL ACCURACY DISTRIBUTION ---')
for score in [3, 2, 1, 0]:
    cnt = (df['factual_accuracy'] == score).sum()
    bar = 'â–ˆ' * cnt
    print(f'  score={score}: {cnt:2d}/{total}  {bar}')

# 3. DB coverage
print(f'\n--- 3. DB COVERAGE ---')
for cov in ['rich', 'sparse', 'missing']:
    cnt = (df['db_coverage'] == cov).sum()
    print(f'  {cov.upper():<8}: {cnt:2d}/{total}')

# 4. Top 5 failing artifacts
print(f'\n--- 4. TOP 5 FAILING ARTIFACTS ---')
worst = df.nsmallest(5, 'factual_accuracy')[['artifact_label','factual_accuracy','hallucination','db_coverage','main_issue']]
for _, r in worst.iterrows():
    print(f'  [{r["factual_accuracy"]}/3] {r["artifact_label"][:50]}')
    print(f'         Coverage={r["db_coverage"]} | Halluc={bool(r["hallucination"])} | {r["main_issue"][:60]}')

# 5. Top 5 recommendations by frequency
print(f'\n--- 5. TOP RECOMMENDATIONS ---')
from collections import Counter
all_recs = []
for lbl in test_labels:
    for rec in extracted.get(lbl, {}).get('recommendations', []):
        all_recs.append(rec)
# Also add judge recommendations
for lbl in test_labels:
    rec = judge_scores.get(lbl, {}).get('recommendation', '')
    if rec and rec != 'none':
        all_recs.append(rec)
for rec, cnt in Counter(all_recs).most_common(5):
    print(f'  ({cnt}x) {rec[:75]}')

print(f'\n{SEP}')
print(f'Analysis complete — {total} artifacts evaluated')
print(SEP)

## Cell 9 — Langfuse Dataset Creation

In [ ]:
DATASET_NAME = 'uguidu-museum-eval'

# Create or get dataset
try:
    dataset = lf.create_dataset(
        name=DATASET_NAME,
        description='UGUIDU museum voice assistant evaluation — Word doc ground truth'
    )
    print(f'Created dataset: {DATASET_NAME}')
except Exception as e:
    print(f'Dataset may already exist: {e}')

item_count = 0
for lbl in test_labels:
    ext = extracted.get(lbl, {})
    sc  = judge_scores.get(lbl, {})
    cov = db_coverage.get(lbl, {})
    try:
        lf.create_dataset_item(
            dataset_name=DATASET_NAME,
            input={'artifact_name': lbl, 'db_search_term': LABEL_TO_DB.get(lbl, '')},
            expected_output={'key_facts': ext.get('key_facts', [])},
            metadata={
                'detection_rate':       ext.get('detection_rate', 'unknown'),
                'rag_quality':          ext.get('rag_quality', 'unknown'),
                'db_records_found':     ext.get('db_records_found', False),
                'llm_response_quality': ext.get('llm_response_quality', 'unknown'),
                'ragtruth_label':       ext.get('ragtruth_label', 'unknown'),
                'main_issues':          ext.get('main_issues', []),
                'recommendations':      ext.get('recommendations', []),
                'judge_factual_accuracy':      sc.get('factual_accuracy', 0),
                'judge_hallucination':         sc.get('hallucination', False),
                'judge_rag_retrieved_correct': sc.get('rag_retrieved_correct', False),
                'db_coverage':          cov.get('coverage', 'missing'),
                'db_hits':              len(cov.get('rows', [])),
            }
        )
        item_count += 1
    except Exception as e:
        log.warning(f'Failed to create dataset item for {lbl}: {e}')

lf.flush()
print(f'Created {item_count} items in Langfuse dataset "{DATASET_NAME}"')
print(f'View at: {LANGFUSE_HOST}/datasets')

## Cell 10 — Export Results

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Sheet 1: All Artifacts ────────────────────────────────────
sheet1_cols = [
    'artifact_label', 'db_search_term', 'doc_artifact_name',
    'detection_rate', 'doc_rag_quality', 'factual_accuracy',
    'hallucination', 'rag_retrieved_correct',
    'key_facts_count', 'issues_count', 'recommendations_count',
    'main_issue', 'recommendation', 'doc_ragtruth_label',
    'llm_response_quality', 'has_doc'
]
sheet1 = df[sheet1_cols].sort_values('factual_accuracy')

# ── Sheet 2: DB Coverage ──────────────────────────────────────
sheet2_rows = []
for lbl, info in db_coverage.items():
    for r in info['rows']:
        sheet2_rows.append({
            'artifact_label': lbl,
            'db_search_term': info['search_term'],
            'coverage': info['coverage'],
            'identifier': r['identifier'],
            'title': r['title'],
            'description_preview': r['description'][:120],
            'content_chars': r['content_len']
        })
    if not info['rows']:
        sheet2_rows.append({
            'artifact_label': lbl,
            'db_search_term': info['search_term'],
            'coverage': 'missing',
            'identifier': '', 'title': '', 'description_preview': '', 'content_chars': 0
        })
sheet2 = pd.DataFrame(sheet2_rows)

# ── Sheet 3: Priority Fixes ───────────────────────────────────
fix_rows = []
for lbl in test_labels:
    ext = extracted.get(lbl, {})
    sc  = judge_scores.get(lbl, {})
    cov = db_coverage.get(lbl, {})
    fa  = sc.get('factual_accuracy', 0)
    impact = 3 - fa  # higher = worse = higher priority
    if cov.get('coverage') == 'missing':
        impact += 2
    if sc.get('hallucination'):
        impact += 1
    for rec in ext.get('recommendations', []):
        fix_rows.append({'artifact_label': lbl, 'impact_score': impact,
                         'recommendation': rec, 'source': 'doc_analysis',
                         'db_coverage': cov.get('coverage', 'missing'),
                         'factual_accuracy': fa})
    judge_rec = sc.get('recommendation', '')
    if judge_rec and judge_rec != 'none':
        fix_rows.append({'artifact_label': lbl, 'impact_score': impact,
                         'recommendation': judge_rec, 'source': 'judge',
                         'db_coverage': cov.get('coverage', 'missing'),
                         'factual_accuracy': fa})
sheet3 = pd.DataFrame(fix_rows).sort_values('impact_score', ascending=False)

# ── Sheet 4: Langfuse Traces ──────────────────────────────────
trace_rows = []
for lbl in test_labels:
    sc = judge_scores.get(lbl, {})
    trace_rows.append({
        'artifact_label': lbl,
        'langfuse_host': LANGFUSE_HOST,
        'traces_url': f'{LANGFUSE_HOST}/traces',
        'dataset_url': f'{LANGFUSE_HOST}/datasets/{DATASET_NAME}',
        'factual_accuracy': sc.get('factual_accuracy', 0),
        'hallucination': sc.get('hallucination', False),
    })
sheet4 = pd.DataFrame(trace_rows)

# ── Export CSV ────────────────────────────────────────────────
csv_path = OUTPUT_DIR / 'eval_with_docs_results.csv'
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'CSV: {csv_path}')

# ── Export Excel ──────────────────────────────────────────────
xlsx_path = OUTPUT_DIR / 'eval_with_docs_summary.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    sheet1.to_excel(writer, sheet_name='All Artifacts', index=False)
    sheet2.to_excel(writer, sheet_name='DB Coverage', index=False)
    sheet3.to_excel(writer, sheet_name='Priority Fixes', index=False)
    sheet4.to_excel(writer, sheet_name='Langfuse Traces', index=False)

    # Auto-width columns for readability
    for sheet in writer.sheets.values():
        for col in sheet.columns:
            max_len = max((len(str(cell.value or '')) for cell in col), default=10)
            sheet.column_dimensions[col[0].column_letter].width = min(max_len + 2, 60)

print(f'Excel: {xlsx_path}')
print(f'\nSheets:')
print(f'  Sheet 1 — All Artifacts:  {len(sheet1)} rows')
print(f'  Sheet 2 — DB Coverage:    {len(sheet2)} rows')
print(f'  Sheet 3 — Priority Fixes: {len(sheet3)} rows')
print(f'  Sheet 4 — Langfuse Traces:{len(sheet4)} rows')
print(f'\nExport complete.')